Cell — load it directly, no upload needed:

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-11-03/ikea.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

(3694, 14)


,Unnamed: 0,item_id,name,category,price,old_price,sellable_online,link,other_colors,short_description,designer,depth,height,width
0,0,90420332,FREKVENS,Bar furniture,265.0,No old price,True,https://www.ikea.com/sa/en/p/frekvens-bar-tabl...,No,"Bar table, in/outdoor, 51x51 cm",Nicholai Wiig Hansen,NaN,99.0,51.0
1,1,368814,NORDVIKEN,Bar furniture,995.0,No old price,False,https://www.ikea.com/sa/en/p/nordviken-bar-tab...,No,"Bar table, 140x80 cm",Francis Cayouette,NaN,105.0,80.0
2,2,9333523,NORDVIKEN / NORDVIKEN,Bar furniture,2095.0,No old price,False,https://www.ikea.com/sa/en/p/nordviken-nordvik...,No,Bar table and 4 bar stools,Francis Cayouette,NaN,NaN,NaN
3,3,80155205,STIG,Bar furniture,69.0,No old price,True,https://www.ikea.com/sa/en/p/stig-bar-stool-wi...,Yes,"Bar stool with backrest, 74 cm",Henrik Preutz,50.0,100.0,60.0
4,4,30180504,NORBERG,Bar furniture,225.0,No old price,True,https://www.ikea.com/sa/en/p/norberg-wall-moun...,No,"Wall-mounted drop-leaf table, ...",Marcus Arvonen,60.0,43.0,74.0


Cleaning cell — handles both issues, filters to a manageable subset, outputs the catalog:

In [ ]:
import re

SAR_TO_USD = 0.27  # approximate, document this assumption

def derive_style(row):
    desc = str(row["short_description"]).lower() + " " + str(row["name"]).lower()
    if any(w in desc for w in ["modern", "minimalist", "sleek"]):
        return "modern"
    if any(w in desc for w in ["rustic", "wood", "oak", "solid wood"]):
        return "traditional"
    if any(w in desc for w in ["glass", "chrome", "metal"]):
        return "industrial"
    return "contemporary"  # default fallback

def derive_color(row):
    desc = str(row["short_description"]).lower()
    for color in ["white", "black", "beige", "brown", "grey", "gray", "oak", "natural"]:
        if color in desc:
            return color
    return "unspecified"

# keep only rows with usable core data
clean_df = df.dropna(subset=["name", "category", "price"]).copy()

clean_df["price_usd"] = (clean_df["price"] * SAR_TO_USD).round(2)
clean_df["style"] = clean_df.apply(derive_style, axis=1)
clean_df["color"] = clean_df.apply(derive_color, axis=1)

# keep manageable size for a portfolio RAG demo — sample per category
catalog_df = clean_df.groupby("category").apply(
    lambda x: x.sample(min(len(x), 15), random_state=42)
).reset_index(drop=True)

print(catalog_df.shape)
catalog_df[["name", "category", "price_usd", "style", "color", "width", "height", "depth"]].head(10)

(253, 17)


/tmp/ipykernel_1114/3578928258.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  catalog_df = clean_df.groupby("category").apply(


,name,category,price_usd,style,color,width,height,depth
0,HENRIKSDAL,Bar furniture,187.65,contemporary,unspecified,40.0,102.0,51.0
1,EKEDALEN / EKEDALEN,Bar furniture,587.52,contemporary,unspecified,NaN,NaN,NaN
2,EKEDALEN,Bar furniture,214.92,contemporary,unspecified,80.0,105.0,NaN
3,STENSELE,Bar furniture,148.50,contemporary,unspecified,NaN,105.0,NaN
4,EKEDALEN,Bar furniture,93.15,contemporary,unspecified,43.0,114.0,51.0
5,NORDVIKEN,Bar furniture,74.25,contemporary,unspecified,40.0,101.0,47.0
6,NORRARYD,Bar furniture,106.65,contemporary,unspecified,46.0,103.0,47.0
7,FRANKLIN,Bar furniture,40.23,contemporary,unspecified,52.0,103.0,44.0
8,NORBERG,Bar furniture,60.75,contemporary,unspecified,74.0,43.0,60.0
9,EKEDALEN,Bar furniture,93.15,contemporary,unspecified,43.0,114.0,52.0


In [ ]:
catalog_df = clean_df.groupby("category", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 15), random_state=42), include_groups=True
).reset_index(drop=True)

/tmp/ipykernel_1114/1227480772.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  catalog_df = clean_df.groupby("category", group_keys=False).apply(


In [ ]:
print(clean_df["short_description"].dropna().sample(15, random_state=1).tolist())

['        Frame with mesh baskets/castors,          25x51x73 cm', '        Coffee table,          75 cm', '        Bench,          132 cm', '        3-seat sofa', '        Chair', '        Table and 4 chairs,          125 cm', '        Storage unit with smart lock,          41x61 cm', '        Frame for table top,          140x140 cm', '        Conference chair', '        Table and 4 chairs,          150 cm', "        Children's armchair", '        Bunk bed frame,          90x200 cm', '        Cabinet,          25x25 cm', '        4-seat sofa', '        Table top,          120x60 cm']


In [ ]:
# known IKEA product line style associations (based on IKEA's own design language per line)
MODERN_LINES = ["malm", "besta", "platsa", "eket", "vihals", "nordli", "kallax", "lack"]
TRADITIONAL_LINES = ["hemnes", "tarva", "leksvik", "idanas", "nordviken", "hurdal"]
INDUSTRIAL_LINES = ["fjallbo", "hilver", "broder"]

def derive_style(row):
    name = str(row["name"]).lower()
    for line in MODERN_LINES:
        if line in name:
            return "modern"
    for line in TRADITIONAL_LINES:
        if line in name:
            return "traditional"
    for line in INDUSTRIAL_LINES:
        if line in name:
            return "industrial"
    # fallback: infer from category text itself
    cat = str(row["category"]).lower()
    if any(w in cat for w in ["sofa", "armchair", "bed"]):
        return "contemporary"
    return "contemporary"

clean_df["style"] = clean_df.apply(derive_style, axis=1)
print(clean_df["style"].value_counts())

style
contemporary    3362
modern           247
traditional       77
industrial         8
Name: count, dtype: int64


In [ ]:
final_columns = ["item_id", "name", "category", "price_usd", "style", "width", "height", "depth"]
catalog_df = clean_df[final_columns].dropna(subset=["price_usd"]).reset_index(drop=True)

catalog_df.to_json("furniture_catalog.json", orient="records", indent=2)
print(catalog_df.shape)
catalog_df.head()

(3694, 8)


,item_id,name,category,price_usd,style,width,height,depth
0,90420332,FREKVENS,Bar furniture,71.55,contemporary,51.0,99.0,NaN
1,368814,NORDVIKEN,Bar furniture,268.65,traditional,80.0,105.0,NaN
2,9333523,NORDVIKEN / NORDVIKEN,Bar furniture,565.65,traditional,NaN,NaN,NaN
3,80155205,STIG,Bar furniture,18.63,contemporary,60.0,100.0,50.0
4,30180504,NORBERG,Bar furniture,60.75,contemporary,74.0,43.0,60.0


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/ai-interior-designer', exist_ok=True)

catalog_df.to_json('/content/drive/MyDrive/ai-interior-designer/furniture_catalog.json', orient="records", indent=2)
print("Saved to Google Drive")

Saved to Google Drive
